## **Elastic Net Regression**

#### **From Scrach**

In [1]:
import numpy as np


class ElasticNetRegression:
    def __init__(
        self,
        alpha=1.0,
        l1_ratio=0.5,
        learning_rate=0.01,
        epochs=1000
    ):
        self.alpha = alpha
        self.l1_ratio = l1_ratio
        self.learning_rate = learning_rate
        self.epochs = epochs

        self.weights = None
        self.bias = 0

    def fit(self, X, y):

        samples, features = X.shape

        self.weights = np.zeros(features)

        for _ in range(self.epochs):

            predictions = (
                np.dot(X, self.weights)
                + self.bias
            )

            error = predictions - y

            l1_penalty = (
                self.alpha
                * self.l1_ratio
                * np.sign(self.weights)
            )

            l2_penalty = (
                self.alpha
                * (1 - self.l1_ratio)
                * self.weights
            )

            dw = (
                (1 / samples)
                * np.dot(X.T, error)
                + l1_penalty
                + l2_penalty
            )

            db = (
                (1 / samples)
                * np.sum(error)
            )

            self.weights -= (
                self.learning_rate * dw
            )

            self.bias -= (
                self.learning_rate * db
            )

    def predict(self, X):
        return (
            np.dot(X, self.weights)
            + self.bias
        )

In [2]:
#Example Usage

X = np.array([
    [1, 2],
    [2, 3],
    [3, 4],
    [4, 5],
    [5, 6]
])

y = np.array([3, 5, 7, 9, 11])

model = ElasticNetRegression(
    alpha=0.1,
    l1_ratio=0.5,
    learning_rate=0.01,
    epochs=5000
)

model.fit(X, y)

predictions = model.predict(X)

print("Weights:")
print(model.weights)

print("Bias:")
print(model.bias)

print("Predictions:")
print(predictions)

Weights:
[0.96637556 0.98486827]
Bias:
0.16104042760418297
Predictions:
[ 3.09715254  5.04839637  6.99964021  8.95088404 10.90212788]


#### **Scikit-Learn Implementation**

In [3]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# 1. Generate synthetic data with correlated features
X, y = make_regression(n_samples=500, n_features=20, noise=0.5, random_state=42)

# 2. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the features (Crucial for Elastic Net regularization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Tune hyperparameters using Cross-Validation (ElasticNetCV)
# l1_ratio: 0.1 is close to Ridge, 0.9 is close to Lasso
alphas = [0.01, 0.1, 1.0, 10.0]
l1_ratios = [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]

cv_model = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, random_state=42, n_jobs=-1)
cv_model.fit(X_train_scaled, y_train)

# 5. Extract the best hyperparameters
best_alpha = cv_model.alpha_
best_l1_ratio = cv_model.l1_ratio
print(f"Optimal Alpha (Strength): {best_alpha}")
print(f"Optimal L1 Ratio (Mix): {best_l1_ratio}")

# 6. Evaluate on the Test Set
y_pred = cv_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

# 7. Check feature sparsity (how many coefficients were set to exactly zero)
zero_coefs = np.sum(cv_model.coef_ == 0)
print(f"Features eliminated by L1 penalty: {zero_coefs} out of {X.shape[1]}")


Optimal Alpha (Strength): 0.01
Optimal L1 Ratio (Mix): [0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0]
Mean Squared Error (MSE): 0.2607
R-squared (R2 Score): 1.0000
Features eliminated by L1 penalty: 2 out of 20
